# The test dataset:

1. https://demo.borealisdata.ca/dataset.xhtml?persistentId=doi%3A10.80240%2FFK2%2F2BWM3O&version=DRAFT
2. https://demo.borealisdata.ca/dataset.xhtml?persistentId=doi%3A10.80240%2FFK2%2FFCZB4A&version=DRAFT

In [4]:
import os
import sys
import yaml
import pandas as pd
import datetime
import janitor # https://pyjanitor.readthedocs.io/
import hashlib
import subprocess
import pydatacuration.utils as utils
import pydatacuration.spell_checker as spell_checker
import pydatacuration.downloads as downloads
import pydatacuration.checksum as checksum
import time
import shutil
import datetime
import dotenv
import importlib

# Clean up the directory enviroment

# Configurate the enviromental variables

In [5]:
# Load the configuration file
with open('config.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [6]:
dotenv.load_dotenv()
BASE_URL = os.getenv('BASE_URL')
API_TOKEN = os.getenv('API_TOKEN')

In [7]:
# Specify the dataset PID
PID = 'doi:10.80240/FK2/FCZB4A'

In [8]:
# The workdir is where the data will be downloaded to
workdir = os.path.join(os.getcwd(), 'workdir')

In [9]:
# Remove the specified directories and their contents
def remove_dirs(dirs):
    for dir in dirs:
        if os.path.exists(dir):
            shutil.rmtree(dir)

remove_dirs(['workdir'])

In [ ]:
# Make the directories
dm = utils.DirectoryManager(workdir)
dm.mk_log_dir()
dm.mk_ds_dir()
dm.mk_temp_dir()

# Download the dataset metadata JSON representation, and data files

In [11]:
# Initiating the downloads
downloads = downloads.Downloads(BASE_URL, API_TOKEN, PID, workdir)

## Metadata JSON

In [12]:
# Download the dataset metadata and export it to a file
ds_metadata = downloads.get_ds_metadata()

## Files

In [13]:
# Download the dataset as a zip file using the 'Basic Download By Dataset' API
ds_zip_path = downloads.get_ds_zip()

In [14]:
# Unzip the file and move the MANIFEST file to the 'dataset/metadata' directory
utils.unzip_file(ds_zip_path, f'{os.path.join(workdir, "dataset", "files")}')

# Compare the files md5 checksum with the JSON file

In [15]:
import pydatacuration.checksum as checksum

In [16]:
# Initialize the checksum module
checksum = checksum.Checksum()

In [17]:
# Get the md5 checksum of the downloaded files
dl_file_checksum_nested_list = checksum.gen_ds_files_checksum(os.path.join(workdir, 'dataset', 'files/'))

## Reterive and parse the data file paths and their md5 checksum

In [18]:
# Extract the file list metadata from the dataset metadata
file_list_metadata = ds_metadata['data']['latestVersion']['files']

In [19]:
# Parse the file list metadata
def parse_file_list_metadata(file_list_metadata):
    file_list_metadata_nested_list = []
    for file_meta in file_list_metadata:
        if file_meta.get('directoryLabel'):
            file_list_metadata_nested_list.append({
                'file': f"{file_meta['directoryLabel']}/{file_meta['dataFile']['filename']}",
                'md5_checksum': file_meta['dataFile']['md5']
            })
        else:
            file_list_metadata_nested_list.append({
                'file': file_meta['dataFile']['filename'],
                'md5_checksum': file_meta['dataFile']['md5']
            })

    return file_list_metadata_nested_list

In [20]:
file_list_metadata_nested_list = parse_file_list_metadata(file_list_metadata)

## Compare the downloaded files md5 and metadata files md5

In [ ]:
utils.compare_files_and_metadata(dl_file_checksum_nested_list, file_list_metadata_nested_list, workdir)

# Perform Checks

## Organization

1. Files and folders are named and structured appropriately (spot check)
   1. Does any file contain special characters? (& , * % . # * ( ) ! @$ ^ ~ ‘ { } [ ] ? < >)?
   2. Does any file name contain more than 32 characters?
   3. Do all the files contain file extensions?

In [106]:
# Initiate the class
FileNameFormatChecker = utils.FileNameFormatChecker()

In [119]:
# 1.1
for file in file_list_metadata:
    if FileNameFormatChecker.check_special_char(file['dataFile']['filename'])[1] == True:
        print(f"Special characters found in the filename: {file['dataFile']['filename']}")


In [120]:
# 1.2

for file in file_list_metadata:
    if FileNameFormatChecker.check_file_name_len(file['dataFile']['filename'], 32)[1] == True:
        print(f"Filename is longer than 32 character: {file['dataFile']['filename']}")

In [ ]:
# 1.3

for file in file_list_metadata:
    if FileNameFormatChecker.check_file_ext(file['dataFile']['filename'])[1] == True:
        print(f"Filename extension not found: {file['dataFile']['filename']}")

2. Data are in standard file formats, or documentation/software information is provided for other formats
   1. Are there any files that do not appear on the Preferred file format list, and no information has been provided to explain how to open the file?

In [ ]:
FileNameFormatChecker.check_flie_preferred_format('res/medium-file-list.txt', 'res/medium-file-list.txt')

## Documentation

1. Does the dataset include a (seperate) README file?

In [26]:
file_name_list = [item.get('dataFile').get('filename') for item in file_list_metadata]

In [ ]:
utils.readme_file_checker('00_README.md')

# Metadata

1. Overall
   1.  Any typo across the metadata fields on dataverse?

The following fields are checked:
1. Title
2. Subtitle
3. Alternative Title
4. Description
5. Notes

## Files

1. Can the files be opened?

In [58]:
# Images


import pydatacuration.files_opener as files_opener

file_opener = files_opener.FilesOpener(f'{workdir}/dataset/files/original.tif')

In [59]:
file_opener.open_image_file()

True

# Output

## Tree-structure of the dataset

In [4]:
# Redirect Tree structure to a text file
original_stdout = sys.stdout # Save a reference to the original standard output
with open(f'{workdir}/log_files/ds_structure.txt', 'w') as f:
    sys.stdout = f # Change the standard output to the file we created.
    utils.list_files('raw_data')
    sys.stdout = original_stdout # Reset the standard output to its original value

## Basic log file

In [15]:
def get_sha256sum(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, 'rb') as f:
        for block in iter(lambda: f.read(4096), b''):
            sha256.update(block)
    return sha256.hexdigest()


def get_hidden_file(file_path):
    return os.path.basename(file_path).startswith('.')

In [16]:
def get_filepaths_and_metadata(directory):
    file_info = []
    id = 1
    for root, dirs, files in os.walk(directory):
        for file in files:
            full_file_path = os.path.join(root, file)
            parent_directory = os.path.basename(os.path.dirname(full_file_path))
            if os.path.exists(full_file_path):
                created = datetime.datetime.fromtimestamp(os.path.getctime(full_file_path))
                modified = datetime.datetime.fromtimestamp(os.path.getmtime(full_file_path))
                file_extension = os.path.splitext(full_file_path)[1]
                sha256_hash = get_sha256sum(full_file_path)
            else:
                created = None
                modified = None
                file_extension = None
                sha256_hash = None
            
            file_info.append({
                'ds.file_id': id,
                'ds.root': directory,
                'ds.parent_directory': parent_directory,
                'ds.depth': full_file_path.count(os.sep) - directory.count(os.sep) + 1,
                'ds.file_name': file,
                'ds.file_path': full_file_path,
                'ds.created': created,
                'ds.modified': modified,
                'ds.file_extension': file_extension,
                'ds.sha256_hash': sha256_hash,
            })
            id += 1
    return file_info

In [ ]:
# Directory to scan
directory = 'raw_data/'

# Get file paths and metadata
file_metadata = get_filepaths_and_metadata(directory)

# Create DataFrame
df = pd.DataFrame(file_metadata)

# Define the order of columns
df = df.reorder_columns(['ds.file_id', 'ds.root'] + [col for col in df.columns if col not in ['ds.file_id', 'ds.root']])

# Convert file_id to integer
df['ds.file_id'] = df['ds.file_id'].astype(int)

In [ ]:
# Export to CSV
df.to_csv('./log_files/ds_file_info.csv', index=False)

## OPF-Fido log

In [ ]:
# Combined command to change directory and run the fido command
command = 'cd ~/MDL/pydatacuration/ && fido -recurse -zip raw_data/ > log_files/temp_data/fileFormats_temp.csv'

# Run the command
subprocess.run(command, shell=True)

# Sleep for 2 seconds to allow the command to finish
time.sleep(2)

In [ ]:
file_formats_df = pd.read_csv('log_files/temp_data/fileFormats_temp.csv')
os.remove('log_files/temp_data/fileFormats_temp.csv')
shutil.rmtree('log_files/temp_data', ignore_errors=True)
file_formats_df.columns = ['fido.status', 'fido.info.time', 'fido.info.puid', 'fido.info.formatname', 'fido.info.signaturename', 'fido.info.filesize', 'fido.info.filename', 'fido.info.mimetype', 'fido.info.matchtype'] 

In [ ]:
# Prepare the DataFrames (if necessary)
df['ds.file_path'] = df['ds.file_path'].str.strip()
file_formats_df['fido.info.filename'] = file_formats_df['fido.info.filename'].str.strip()

# Perform a left join
dataCuration_df = pd.merge(df, file_formats_df, left_on='ds.file_path', right_on='fido.info.filename', how='left')

In [ ]:
# Check wehther the directory contains blank spaces
dataCuration_df['cur.dir.blank'] = dataCuration_df['ds.file_path'].apply(lambda x: 'TRUE' if ' ' in os.path.dirname(x) else 'FALSE')

# Check whether the file contains blank spaces
dataCuration_df['cur.file_name.blank'] = dataCuration_df['ds.file_name'].apply(lambda x: 'TRUE' if ' ' in x else 'FALSE')

# Check whether the file is a hidden file
dataCuration_df['cur.file.hidden'] = dataCuration_df['ds.file_path'].apply(lambda x: 'TRUE' if get_hidden_file(x) else 'FALSE')

In [ ]:
# Remove the duplicate column
dataCuration_df.drop(columns=['fido.info.filename'], inplace=True)

# Export to CSV
dataCuration_df.to_csv('./log_files/dataCuration.csv', index=False)

In [ ]:
with open('./log_files/ds_structure.txt', 'r') as f:
    ds_structure_file_content = f.read()

In [ ]:
markdown_text = f"""
# Project information

- Ticket Number: {config['project']['ticket-number']}

- Data curator: {config['curator']['name']}

- Data curator email: {config['curator']['email']}

- Date: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

# Submission details

- Data submitter: {config['submission']['submitter']['name']}

- Contact: {config['submission']['submitter']['email']}

- Date: {config['submission']['date']}


# Dataset details:

- Number of files: {df['ds.file_id'].count()}

- Total size: {df['ds.file_path'].apply(lambda x: os.path.getsize(x)).sum()/(1024*1024):.2f} MB


# File list
{dataCuration_df.to_markdown()}

# Dataset Structure
```
{ds_structure_file_content}
```

# Associated files
1. [Basic File Information](./log_files/ds_file_info.csv)
2. [Detailed File Information](./log_files/dataCuration.csv)
3. [Dataset Structure](./log_files/ds_structure.txt)

"""

In [ ]:
with open('./log_files/output.md', 'w') as f:
    f.write(markdown_text)

# Query